In [1]:
# Импорт необходимых библиотек
import sys
import os
from pathlib import Path

# Добавляем src в путь для импорта модулей
sys.path.append(str(Path.cwd().parent / "src"))

import torch
import numpy as np
import pandas as pd

# Импорт наших модулей
from config import Config
from load_data import load_data
from tools import build_barrier_labels, make_windows
from model import create_model, create_optimizer_and_scheduler
from dataset import create_data_loaders
from trainer import create_trainer

print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA версия: {torch.version.cuda}")
print(f"Устройство: {'cuda' if torch.cuda.is_available() else 'cpu'}")

ModuleNotFoundError: No module named 'config'

In [ ]:
# Создание конфигурации
config = Config.default()

# Настройка параметров для нескольких дней
config.data.train_dates = ["2025-09-22", "2025-09-23"]  # Список дней для обучения
config.data.val_date = "2025-09-24"                     # День для валидации
config.data.test_date = "2025-09-25"                    # День для тестирования
config.data.data_folder = "../data"

# Параметры модели
config.model.tick_size = 1.0
config.model.theta_ticks = 5
config.model.horizon_sec = 2.0
config.model.window_length = 240
config.model.use_cost_sensitive = True  # Использовать стоимостно-чувствительный лосс

print("Конфигурация создана:")
print(f"Дни обучения: {config.data.train_dates}")
print(f"День валидации: {config.data.val_date}")
print(f"День тестирования: {config.data.test_date}")
print(f"Папка с данными: {config.data.data_folder}")
print(f"Эпохи на день: {config.training.epochs}")
print(f"Функция потерь: {'Cost-Sensitive Loss' if config.model.use_cost_sensitive else 'Focal Loss'}")

data/Si-12.25_2025-09-22_features.npy
Форма массива: (143799, 138)
Тип данных: float32


In [ ]:
# Загрузка и подготовка данных для обучения
print("Загрузка данных для обучения...")
train_data_loaders = []

for i, train_date in enumerate(config.data.train_dates):
    print(f"\nОбработка дня обучения {i+1}/{len(config.data.train_dates)}: {train_date}")
    
    # Загрузка данных
    features_file = config.data.get_features_file(train_date)
    prices_file = config.data.get_prices_file(train_date)
    X, ms, mid = load_data(features_file, prices_file)
    
    # Построение меток и окон
    y_all = build_barrier_labels(
        ms, mid, 
        tick_size=config.model.tick_size,
        theta_ticks=config.model.theta_ticks, 
        horizon_sec=config.model.horizon_sec
    )
    
    # Фильтрация валидных меток
    valid = (y_all != -1)
    Xv, yv, msv, midv = X[valid], y_all[valid], ms[valid], mid[valid]
    
    # Создание окон
    Xwin, end_idx = make_windows(Xv, config.model.window_length)
    y_win = yv[end_idx]
    
    print(f"  Классы и счётчики: {np.unique(y_win, return_counts=True)}")
    print(f"  Окна: {Xwin.shape} | метки: {y_win.shape}")
    
    # Создание DataLoader для этого дня
    from dataset import NpWindowDataset, BalancedBatchSampler, BalancedBatchBatchSampler
    from torch.utils.data import DataLoader
    
    train_ds = NpWindowDataset(Xwin, y_win)
    base_sampler = BalancedBatchSampler(
        y_win, 
        batch_size=config.training.batch_size, 
        num_classes=config.model.num_classes, 
        seed=config.training.seed
    )
    batch_sampler = BalancedBatchBatchSampler(base_sampler, config.training.batch_size)
    train_dl = DataLoader(train_ds, batch_sampler=batch_sampler)
    train_data_loaders.append(train_dl)

In [ ]:

# Загрузка данных для валидации
print(f"\nЗагрузка данных для валидации: {config.data.val_date}")
val_features_file = config.data.get_features_file(config.data.val_date)
val_prices_file = config.data.get_prices_file(config.data.val_date)
X_val, ms_val, mid_val = load_data(val_features_file, val_prices_file)

y_val_all = build_barrier_labels(
    ms_val, mid_val, 
    tick_size=config.model.tick_size,
    theta_ticks=config.model.theta_ticks, 
    horizon_sec=config.model.horizon_sec
)

valid_val = (y_val_all != -1)
Xv_val, yv_val = X_val[valid_val], y_val_all[valid_val]
Xwin_val, end_idx_val = make_windows(Xv_val, config.model.window_length)
y_win_val = yv_val[end_idx_val]

val_ds = NpWindowDataset(Xwin_val, y_win_val)
val_dl = DataLoader(val_ds, batch_size=config.training.batch_size, shuffle=False)

print(f"Валидация: {Xwin_val.shape} | метки: {y_win_val.shape}")

# Загрузка данных для тестирования
print(f"\nЗагрузка данных для тестирования: {config.data.test_date}")
test_features_file = config.data.get_features_file(config.data.test_date)
test_prices_file = config.data.get_prices_file(config.data.test_date)
X_test, ms_test, mid_test = load_data(test_features_file, test_prices_file)

y_test_all = build_barrier_labels(
    ms_test, mid_test, 
    tick_size=config.model.tick_size,
    theta_ticks=config.model.theta_ticks, 
    horizon_sec=config.model.horizon_sec
)

valid_test = (y_test_all != -1)
Xv_test, yv_test = X_test[valid_test], y_test_all[valid_test]
Xwin_test, end_idx_test = make_windows(Xv_test, config.model.window_length)
y_win_test = yv_test[end_idx_test]

test_ds = NpWindowDataset(Xwin_test, y_win_test)
test_dl = DataLoader(test_ds, batch_size=config.training.batch_size, shuffle=False)

print(f"Тестирование: {Xwin_test.shape} | метки: {y_win_test.shape}")

In [ ]:
print("Все DataLoader'ы созданы успешно!")
print(f"Количество дней обучения: {len(train_data_loaders)}")
print(f"Валидационный DataLoader: {len(val_dl)} батчей")
print(f"Тестовый DataLoader: {len(test_dl)} батчей")


In [ ]:
# Создание модели
model, criterion = create_model(config.model)
optimizer, scheduler, scaler = create_optimizer_and_scheduler(model, config.training)

model = model.to(config.training.device)

print(f"Модель создана: {sum(p.numel() for p in model.parameters())} параметров")
print(f"Устройство: {config.training.device}")


In [ ]:
# Создание тренера и последовательное обучение
trainer = create_trainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    device=config.training.device,
    grad_clip_norm=config.training.grad_clip_norm
)

# Последовательное обучение на нескольких днях
print("Начало последовательного обучения...")
training_results = trainer.train_sequential_days(
    train_data_loaders=train_data_loaders,
    val_loader=val_dl,
    epochs_per_day=config.training.epochs,
    save_path_template="model_day_{}.pt",
    verbose=True
)

print(f"\nЛучший macro-F1 на валидации: {training_results['best_score']:.4f}")


In [ ]:
# Тестирование и сохранение модели
test_results = trainer.test(test_dl)

# Сохранение финальной модели
model_path = "best_deeplob_like.pt"
trainer.save_model(model_path)

print(f"\nОбучение завершено!")
print(f"Финальная модель сохранена: {os.path.abspath(model_path)}")
print(f"Промежуточные модели сохранены: model_day_1.pt, model_day_2.pt, ...")
